In [1]:
import sys
sys.path.append('..') # Para poder importar desde la carpeta src
from src.processor import procesar_csv_brou, aplicar_reglas
from src.database import get_connection
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.express as px

# 1. Cargar el archivo más reciente de data/raw/
import glob
files = glob.glob("../data/raw/*.csv")
latest_file = max(files, key=os.path.getctime)
df_nuevo = procesar_csv_brou(latest_file)
df_con_reglas = aplicar_reglas(df_nuevo)

NameError: name 'os' is not defined

In [ ]:
pendientes = df_con_reglas[df_con_reglas['categoria'].isnull()]
categorias = ["Super", "Fijos", "Ocio", "Transporte", "Salud", "Educación"]

if not pendientes.empty:
    idx = pendientes.index[0]
    row = pendientes.iloc[0]
    
    print(f"Gasto desconocido: {row['descripcion']} - Monto: {row['monto']}")
    selector = widgets.Dropdown(options=categorias, description='Categoría:')
    boton = widgets.Button(description="Guardar Regla", button_style='warning')
    
    def guardar_y_seguir(b):
        conn = get_connection()
        # Guardamos los primeros 12 caracteres como palabra clave
        keyword = row['descripcion'][:12] 
        conn.execute("INSERT INTO reglas (keyword, categoria) VALUES (?, ?)", (keyword, selector.value))
        conn.commit()
        conn.close()
        clear_output()
        print("Regla guardada. Ejecuta esta celda de nuevo.")
        
    boton.on_click(guardar_y_seguir)
    display(selector, boton)
else:
    print("✅ Todo categorizado. Listo para guardar en BD.")

In [ ]:
# Guardar en base de datos (ignora duplicados por el hash)
conn = get_connection()
df_con_reglas.to_sql("movimientos", conn, if_exists="append", index=False, method="multi")

# Cargar todo para el gráfico
df_total = pd.read_sql("SELECT * FROM movimientos", conn)
fig = px.pie(df_total, values='monto', names='categoria', title='Mis Gastos Totales')
fig.show()
conn.close()